In [1]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import numpy as np
import clahe 

I0000 00:00:1789726418.077711    1151 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789726419.209980    1151 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789726422.365178    1151 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
training_path = "../Datasets/data/train/Medical/X-Ray/shoulder/Train"
testing_path = "../Datasets/data/test/Medical/X-Ray/shoulder/Test"
valid_path = "../Datasets/data/vaild/Medical/X-Ray/shoulder/valid"

In [3]:
train_dataug = ImageDataGenerator(
    preprocessing_function = lambda img :clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    zoom_range = 0.2,
    horizontal_flip = True
)
test_dataug = ImageDataGenerator(
    preprocessing_function = lambda img :clahe.clahe_preprocess(img,preprocess_input)
)
valid_dataug = ImageDataGenerator(
    preprocessing_function = lambda img :clahe.clahe_preprocess(img,preprocess_input)
)

In [4]:
training_set = train_dataug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size = 32,
    class_mode = "binary"
)
valid_set = valid_dataug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 32,
    class_mode = "binary"
)
testing_set = test_dataug.flow_from_directory(
    testing_path,
    target_size = (224,224),
    batch_size = 32,
    class_mode = "binary",
    shuffle = False
)

Found 7690 images belonging to 2 classes.
Found 973 images belonging to 2 classes.
Found 1542 images belonging to 2 classes.


In [5]:
training_set.class_indices

{'abnormal': 0, 'normal': 1}

In [6]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

I0000 00:00:1789726429.300721    1151 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [7]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(2556,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
output = Dense(1,activation="sigmoid")(x)

In [8]:
model = Model(inputs = base_model.input,outputs = output)

In [9]:
model.compile(optimizer=Adam(learning_rate=0.002),metrics=["accuracy"],loss="binary_crossentropy")

In [10]:
early_stop = EarlyStopping(monitor="loss",restore_best_weights=True,patience=4)

In [11]:
model.fit(training_set,validation_data=(valid_set),epochs=15,callbacks=[early_stop])

Epoch 1/15


I0000 00:00:1789726433.190700    1151 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789726441.576682    2001 service.cc:153] XLA service 0x7ea524035b50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789726441.576761    2001 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1789726441.963556    2001 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789726443.579636    2001 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1789726444.352919    2001 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_19157__.265
E0000 00:00:1789726456.969965    2001 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, 

171/241 ━━━━━━━━━━━━━━━━━━━━ 29s 421ms/step - accuracy: 0.5847 - loss: 0.7684

I0000 00:00:1789726546.515236    1999 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_19157__.265


241/241 ━━━━━━━━━━━━━━━━━━━━ 191s 632ms/step - accuracy: 0.6506 - loss: 0.6481 - val_accuracy: 0.7163 - val_loss: 0.5819
Epoch 2/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 84s 349ms/step - accuracy: 0.7098 - loss: 0.5764 - val_accuracy: 0.7318 - val_loss: 0.5544
Epoch 3/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 88s 366ms/step - accuracy: 0.7229 - loss: 0.5595 - val_accuracy: 0.7133 - val_loss: 0.5640
Epoch 4/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 88s 366ms/step - accuracy: 0.7229 - loss: 0.5493 - val_accuracy: 0.7235 - val_loss: 0.5595
Epoch 5/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 72s 297ms/step - accuracy: 0.7380 - loss: 0.5313 - val_accuracy: 0.7235 - val_loss: 0.5413
Epoch 6/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 81s 337ms/step - accuracy: 0.7445 - loss: 0.5302 - val_accuracy: 0.7338 - val_loss: 0.5376
Epoch 7/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 91s 378ms/step - accuracy: 0.7480 - loss: 0.5241 - val_accuracy: 0.7482 - val_loss: 0.5188
Epoch 8/15
241/241 ━━━━━━━━━━━━━━━━━━━━ 87s 361ms/step - accuracy: 0.7524 - loss: 0.5198 - va

In [12]:
prediction = model.predict(testing_set)
true_prediction = (prediction > 0.5).astype(int)

49/49 ━━━━━━━━━━━━━━━━━━━━ 24s 408ms/step


In [13]:
true_prediction

array([[0],
       [0],
       [0],
       ...,
       [1],
       [1],
       [1]], shape=(1542, 1))

In [14]:
y_true = testing_set.classes

In [15]:
print(classification_report(y_true,true_prediction))
confusion_matrix(y_true,true_prediction)

              precision    recall  f1-score   support

           0       0.74      0.74      0.74       752
           1       0.75      0.75      0.75       790

    accuracy                           0.75      1542
   macro avg       0.75      0.75      0.75      1542
weighted avg       0.75      0.75      0.75      1542



array([[555, 197],
       [195, 595]])

In [16]:
model1 = model

In [17]:
# model1.save('dsff.keras')

In [18]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False


In [19]:
model.compile(optimizer=Adam(learning_rate=1e-5),loss="binary_crossentropy",metrics=["accuracy"])

In [20]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [21]:
history = model.fit(training_set,validation_data=valid_set,epochs=30,callbacks=[early_stop, reduce_lr])

Epoch 1/30


I0000 00:00:1789727766.496935    2002 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_77962__.289


101/241 ━━━━━━━━━━━━━━━━━━━━ 34s 245ms/step - accuracy: 0.7046 - loss: 0.5972

I0000 00:00:1789727808.106389    2001 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_77962__.289


241/241 ━━━━━━━━━━━━━━━━━━━━ 112s 360ms/step - accuracy: 0.7306 - loss: 0.5499 - val_accuracy: 0.7215 - val_loss: 0.5794 - learning_rate: 1.0000e-05
Epoch 2/30
241/241 ━━━━━━━━━━━━━━━━━━━━ 66s 275ms/step - accuracy: 0.7429 - loss: 0.5210 - val_accuracy: 0.7287 - val_loss: 0.5659 - learning_rate: 1.0000e-05
Epoch 3/30
241/241 ━━━━━━━━━━━━━━━━━━━━ 72s 300ms/step - accuracy: 0.7534 - loss: 0.5088 - val_accuracy: 0.7369 - val_loss: 0.5517 - learning_rate: 1.0000e-05
Epoch 4/30
241/241 ━━━━━━━━━━━━━━━━━━━━ 80s 333ms/step - accuracy: 0.7583 - loss: 0.5007 - val_accuracy: 0.7348 - val_loss: 0.5508 - learning_rate: 1.0000e-05
Epoch 5/30
241/241 ━━━━━━━━━━━━━━━━━━━━ 71s 293ms/step - accuracy: 0.7628 - loss: 0.4926 - val_accuracy: 0.7410 - val_loss: 0.5450 - learning_rate: 1.0000e-05
Epoch 6/30
241/241 ━━━━━━━━━━━━━━━━━━━━ 80s 331ms/step - accuracy: 0.7610 - loss: 0.4974 - val_accuracy: 0.7441 - val_loss: 0.5417 - learning_rate: 1.0000e-05
Epoch 7/30
241/241 ━━━━━━━━━━━━━━━━━━━━ 74s 306ms/step -

In [22]:
result = model.predict(testing_set)

49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 253ms/step


In [23]:
result = (result >= 0.5).astype(int)

In [24]:
print(classification_report(y_true,result))
print(confusion_matrix(y_true,result))

              precision    recall  f1-score   support

           0       0.80      0.70      0.74       752
           1       0.74      0.83      0.78       790

    accuracy                           0.76      1542
   macro avg       0.77      0.76      0.76      1542
weighted avg       0.77      0.76      0.76      1542

[[523 229]
 [134 656]]
